# Concat Only Resting/Thinking Normal Task CSVs

This notebook concatenates only these files:
- `resting_mav_normal - *.csv`
- `resting_var_normal - *.csv`
- `thinking_mav_normal - *.csv`
- `thinking_var_normal - *.csv`

It excludes `ERD_ERS` and `encoding chaining.csv`, keeps all values as text, and ensures the final output contains no NaN values.

In [ ]:
import pandas as pd
from pathlib import Path
import re

ROOT = Path('.').resolve()
OUTPUT_CSV = ROOT / 'normal_task_features_concat_only.csv'
FILE_PATTERN = re.compile(
    r'^(resting|thinking)_(mav|var)_normal\s*-\s*(c3|c4|cz)\.csv$',
    flags=re.IGNORECASE,
)

print(f'Working directory: {ROOT}')
print(f'Output CSV: {OUTPUT_CSV}')

In [ ]:
def parse_task_meta(path: Path):
    match = FILE_PATTERN.match(path.name)
    if match is None:
        return None

    task_raw, feature_raw, channel_raw = match.groups()
    return {
        'task': task_raw.lower(),
        'feature_type': 'mav' if feature_raw.lower() == 'mav' else 'variance',
        'channel': channel_raw.upper(),
    }

target_files = []
for path in sorted(ROOT.glob('*.csv')):
    lower_name = path.name.lower()
    if lower_name == 'encoding chaining.csv':
        continue
    if 'erd_ers' in lower_name:
        continue

    meta = parse_task_meta(path)
    if meta is not None:
        target_files.append((path, meta))

if not target_files:
    raise FileNotFoundError('No resting/thinking mav/var normal CSV files were found.')

print(f'Files to concatenate: {len(target_files)}')
for path, meta in target_files:
    print(f"- {path.name} | task={meta['task']} | feature={meta['feature_type']} | channel={meta['channel']}")

In [ ]:
frames = []
for path, meta in target_files:
    # Read as text and disable NA parsing so tiny/zero values are preserved exactly.
    df = pd.read_csv(
        path,
        header=None,
        dtype=str,
        keep_default_na=False,
        na_filter=False,
        encoding='utf-8-sig',
    )

    df.columns = [f'col_{i + 1}' for i in range(df.shape[1])]
    df.insert(0, 'source_row', range(1, len(df) + 1))
    df.insert(0, 'channel', meta['channel'])
    df.insert(0, 'feature_type', meta['feature_type'])
    df.insert(0, 'task', meta['task'])
    df.insert(0, 'source_file', path.name)

    frames.append(df)

combined = pd.concat(frames, ignore_index=True, sort=False)
combined = combined.fillna('')

nan_total = int(combined.isna().sum().sum())
if nan_total != 0:
    raise ValueError(f'NaN values detected after concatenation: {nan_total}')

print(f'Combined rows: {len(combined):,}')
print(f'Combined columns: {combined.shape[1]:,}')
print(f'Total NaN values: {nan_total}')
display(combined.head(10))

In [ ]:
combined.to_csv(OUTPUT_CSV, index=False)
print(f'Saved: {OUTPUT_CSV}')

rows_per_file = (
    combined.groupby(['source_file', 'task', 'feature_type', 'channel'], as_index=False)
    .size()
    .rename(columns={'size': 'rows'})
)
display(rows_per_file)